# Session C — conflict seasonality figures

Run: `python -m archive.impact_eda.impact_session_c --through 26`

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from processing_data.paths import COURSE_RAW, EXT_DATA

ROOT = Path.cwd()
if not (ROOT / "archive" / "impact_eda" / "impact_session_c.py").exists():
    ROOT = ROOT.parents[2]
OUT = ROOT / "eda" / "outputs" / "impact_eda"
FIG = OUT / "figures" / "session_c"
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    p = FIG / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", p.relative_to(ROOT))

In [ ]:
# Fig 1 — dominant livelihood (pastoral outlined)
lhz = gpd.read_file(EXT_DATA / "SS_LHZ_2018" / "SS_LHZ_2018.shp")
dom = pd.read_csv(OUT / "stage23_admin2_lhz.csv")
past_codes = set(dom.loc[dom.pastoral_dominant, "LZCODE"])
fig, ax = plt.subplots(figsize=(7, 8))
lhz[~lhz.LZCODE.isin(past_codes)].plot(ax=ax, color="#d5e8d4", edgecolor="gray", linewidth=0.5)
lhz[lhz.LZCODE.isin(past_codes)].plot(ax=ax, color="#f9e79f", edgecolor="#c0392b", linewidth=1.2)
ax.set_title("FEWS livelihood zones 2018; gold = pastoral/cattle/livestock dominant counties")
ax.axis("off")
save(fig, "fig01_livelihood_zones.png")

In [ ]:
# Fig 2 — H8 wet vs dry-lead coefs (spec all)
ols = pd.read_csv(OUT / "stage24_h8_seasonal_ols.csv")
sub = ols[(ols.spec == "all") & ols.term.str.contains("flood")]
labels = sub.model + "\n" + sub.spec
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(labels, sub.coef, color=["#3498db", "#e67e22"])
ax.axvline(0, color="gray")
for i, (_, r) in enumerate(sub.iterrows()):
    ax.text(r.coef, i, f" p={r.pvalue:.3f}", va="center", fontsize=9)
ax.set_xlabel("Within FE coef on log flood")
ax.set_title("SC-H1: wet concurrent vs dry-lead type-2 (spec=all)")
save(fig, "fig02_h8_wet_dry_coefs.png")

In [ ]:
# Fig 3 — monthly profiles (national sum)
ged = pd.read_csv(OUT / "stage14_ged_admin2_month.csv")
flood = pd.read_csv(OUT / "flood_admin2_month.csv")
u = flood[flood.mask_type == "unusual"]
g = ged.groupby("month", observed=True)["n_events_type2"].sum()
f = u.groupby("month", observed=True)["pixel_days"].sum()
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(g.index - 0.15, g.values, width=0.3, label="GED type-2 events", color="#7f8c8d")
ax1.set_ylabel("National type-2 events (sum)")
ax2 = ax1.twinx()
ax2.bar(f.index + 0.15, f.values / 1e6, width=0.3, label="unusual flood px-days (M)", color="#2980b9", alpha=0.7)
ax2.set_ylabel("Unusual pixel-days (millions)")
ax1.set_xlabel("Month")
ax1.set_title("Seasonality context: violence vs flood detections")
fig.legend(loc="upper right")
save(fig, "fig03_monthly_profiles.png")

In [ ]:
# Fig 4 — DTM distance distributions
sites = pd.read_csv(OUT / "stage25_dtm_site_distances.csv")
dis = sites[sites.any_disaster]["dist_km_nearest_unusual_flood"]
con = sites[sites.any_conflict]["dist_km_nearest_unusual_flood"]
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(dis, bins=30, alpha=0.6, label=f"disaster arrival (n={len(dis)})", density=True)
ax.hist(con, bins=30, alpha=0.6, label=f"conflict arrival (n={len(con)})", density=True)
ax.set_xlabel("km to nearest unusual flood pixel (2022-24)")
ax.set_title("SC-H2: DTM site distance by arrival reason")
ax.legend()
save(fig, "fig04_dtm_distances.png")

In [ ]:
# Fig 5 — county disaster share vs flood; point size = IDP stock
cm = pd.read_csv(OUT / "stage25_county_dtm_flood.csv")
admin2 = gpd.read_file(COURSE_RAW / "Administrative boundaries" / "ssd_admin2.geojson")[["adm2_pcode", "adm2_name"]]
cm = cm.merge(admin2, on="adm2_pcode", how="left")
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(cm.mean_unusual_px, cm.disaster_share, s=np.clip(cm.idp_stock / 500, 10, 200), alpha=0.7)
ax.set_xlabel("Mean unusual flood unique px (2022-24)")
ax.set_ylabel("Disaster arrival share (DTM sites)")
ax.set_title("County: disaster-tagged arrivals vs flood extent (size ~ stock)")
save(fig, "fig05_county_disaster_vs_flood.png")